# Models


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## U-Net Architecture
* Analogiclly to the img, but adjusted for 1 channel input:

In [3]:
class DoubleConvBN(nn.Module):
    """
    Conv(3x3) -> BatchNorm -> ReLU -> Conv(3x3) -> BatchNorm -> ReLU
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

In [4]:
class UNet(nn.Module):
    """
    Original U-Net (with DoubleConvBN blocks):

    Encoder: 64, 128, 256, 512
    Bottleneck: 1024
    Decoder: 512, 256, 128, 64
    """
    def __init__(self, in_channels=1, n_classes=5):
        super().__init__()

        # -------- Encoder --------
        self.enc1 = DoubleConvBN(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConvBN(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConvBN(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConvBN(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # -------- Bottleneck --------
        self.bottleneck = DoubleConvBN(512, 1024)

        # -------- Decoder --------
        self.up1  = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec1 = DoubleConvBN(1024, 512)   # 512 (up) + 512 (skip)

        self.up2  = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DoubleConvBN(512, 256)    # 256 + 256

        self.up3  = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConvBN(256, 128)    # 128 + 128

        self.up4  = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec4 = DoubleConvBN(128, 64)     # 64 + 64

        self.out_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)      # [B,64,256,256]
        p1 = self.pool1(x1)    # [B,64,128,128]

        x2 = self.enc2(p1)     # [B,128,128,128]
        p2 = self.pool2(x2)    # [B,128,64,64]

        x3 = self.enc3(p2)     # [B,256,64,64]
        p3 = self.pool3(x3)    # [B,256,32,32]

        x4 = self.enc4(p3)     # [B,512,32,32]
        p4 = self.pool4(x4)    # [B,512,16,16]

        b  = self.bottleneck(p4)  # [B,1024,16,16]

        u1 = self.up1(b)       # [B,512,32,32]
        u1 = torch.cat([u1, x4], dim=1)
        u1 = self.dec1(u1)     # [B,512,32,32]

        u2 = self.up2(u1)      # [B,256,64,64]
        u2 = torch.cat([u2, x3], dim=1)
        u2 = self.dec2(u2)     # [B,256,64,64]

        u3 = self.up3(u2)      # [B,128,128,128]
        u3 = torch.cat([u3, x2], dim=1)
        u3 = self.dec3(u3)     # [B,128,128,128]

        u4 = self.up4(u3)      # [B,64,256,256]
        u4 = torch.cat([u4, x1], dim=1)
        u4 = self.dec4(u4)     # [B,64,256,256]

        out = self.out_conv(u4)  # [B, n_classes, 256, 256]
        return out


## Res U-Net Architecture 

* The Res-UNet is an extension of the UNet that incorporates residual connections.
* Residual connections have been introduced in the context of residual networks (ResNets) to address the vanishing gradient problem in deep networks.
* The Res-UNet combines the strengths of the UNet with the benefits of residual connections.
* The convolution block in the UNet is replaced here with residual blocks, and this introduces an addition layer between the input at each block and the output from the last 3x3 convolutional block.

In [5]:
class ResidualBlock(nn.Module):
    """
    Basic ResNet-style block:
    x -> Conv(3x3)-BN-ReLU -> Conv(3x3)-BN -> + shortcut -> ReLU
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        # 1x1 conv if we need to match channel dimensions for the residual
        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + residual
        out = self.relu(out)
        return out


class ResUNet(nn.Module):
    """
    Res-UNet matching the architecture in the figure:

    - 4 encoder stages:   64, 128, 256, 512
    - bottleneck stage:   1024
    - 4 decoder stages, with skip connections
    - residual blocks at each stage
    """
    def __init__(self, in_channels=1, n_classes=5, base_c=64):
        super().__init__()

        # ----- Encoder -----
        self.enc1 = ResidualBlock(in_channels,      base_c)        # 64
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ResidualBlock(base_c,          base_c * 2)     # 128
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ResidualBlock(base_c * 2,      base_c * 4)     # 256
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = ResidualBlock(base_c * 4,      base_c * 8)     # 512
        self.pool4 = nn.MaxPool2d(2)

        # ----- Bottleneck -----
        self.bottleneck = ResidualBlock(base_c * 8, base_c * 16)   # 1024

        # ----- Decoder -----
        self.up1  = nn.ConvTranspose2d(base_c * 16, base_c * 8, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock(base_c * 16,      base_c * 8)

        self.up2  = nn.ConvTranspose2d(base_c * 8,  base_c * 4, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(base_c * 8,       base_c * 4)

        self.up3  = nn.ConvTranspose2d(base_c * 4,  base_c * 2, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(base_c * 4,       base_c * 2)

        self.up4  = nn.ConvTranspose2d(base_c * 2,  base_c,     kernel_size=2, stride=2)
        self.dec4 = ResidualBlock(base_c * 2,       base_c)

        # ----- Final 1x1 conv -----
        self.out_conv = nn.Conv2d(base_c, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)      # [B, base_c,   H,   W]
        p1 = self.pool1(x1)    # [B, base_c,   H/2, W/2]

        x2 = self.enc2(p1)     # [B, 2*base_c, H/2, W/2]
        p2 = self.pool2(x2)    # [B, 2*base_c, H/4, W/4]

        x3 = self.enc3(p2)     # [B, 4*base_c, H/4, W/4]
        p3 = self.pool3(x3)    # [B, 4*base_c, H/8, W/8]

        x4 = self.enc4(p3)     # [B, 8*base_c, H/8, W/8]
        p4 = self.pool4(x4)    # [B, 8*base_c, H/16, W/16]

        # Bottleneck
        b = self.bottleneck(p4)   # [B, 16*base_c, H/16, W/16]

        # Decoder
        u1 = self.up1(b)          # [B, 8*base_c, H/8, W/8]
        u1 = torch.cat([u1, x4], dim=1)
        u1 = self.dec1(u1)        # [B, 8*base_c, H/8, W/8]

        u2 = self.up2(u1)         # [B, 4*base_c, H/4, W/4]
        u2 = torch.cat([u2, x3], dim=1)
        u2 = self.dec2(u2)        # [B, 4*base_c, H/4, W/4]

        u3 = self.up3(u2)         # [B, 2*base_c, H/2, W/2]
        u3 = torch.cat([u3, x2], dim=1)
        u3 = self.dec3(u3)        # [B, 2*base_c, H/2, W/2]

        u4 = self.up4(u3)         # [B, base_c, H, W]
        u4 = torch.cat([u4, x1], dim=1)
        u4 = self.dec4(u4)        # [B, base_c, H, W]

        out = self.out_conv(u4)   # [B, n_classes, H, W]
        return out


## Attention Res U-Net

* As illustrated in Figure, the attention Res-UNet model is built on the Res-UNet architecture, which intro- duces spatial attention mechanisms. 
* This is achieved through the gating signal which brings output from the lower layer to match the same dimension as the current layer, and an attention block which combines information from two sources: the input feature map (x) and the gating signal (gating) to compute attention weights. 
* The specific attention mechanism used here is the spatial attention, where the attention weights are computed based on relationships between different spatial regions in the input feature map. This allows the model to focus on salient parts of the image while suppressing irrelevant regions. 


In [6]:

# -------------------------
# Residual block
# -------------------------
class ResidualBlock(nn.Module):
    """
    Res block:
      x -> Conv3x3-BN-ReLU -> Conv3x3-BN -> + shortcut -> ReLU
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)

        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + res
        out = self.relu(out)
        return out


# -------------------------
# Attention Gate (from Attention U-Net)
# -------------------------
class AttentionGate(nn.Module):
    """
    Attention gate for a skip connection.

    Inputs:
      x: skip feature map from encoder (shape [B, F_l, H, W])
      g: gating signal from decoder (shape [B, F_g, H, W])  (same spatial size)

    Output:
      x_att: attended skip features (same shape as x)
    """
    def __init__(self, F_l: int, F_g: int, F_int: int):
        super().__init__()

        # theta_x: 1x1 conv to intermediate channels
        self.theta_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int),
        )

        # phi_g: 1x1 conv to intermediate channels
        self.phi_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int),
        )

        # psi: 1x1 conv to 1 channel attention coefficients
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        # both are [B, *, H, W]
        theta_x = self.theta_x(x)
        phi_g   = self.phi_g(g)

        f = self.relu(theta_x + phi_g)
        alpha = self.psi(f)            # [B,1,H,W]
        return x * alpha               # broadcast over channels


# -------------------------
# Up block: upconv + attention + concat + residual
# -------------------------
class UpBlockAttention(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        """
        in_ch:   channels coming from below (decoder input)
        skip_ch: channels in encoder skip feature
        out_ch:  channels after upsampling (also decoder stage channels)
        """
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)

        # attention gate uses:
        # x = skip (skip_ch), g = upsampled decoder (out_ch)
        self.att = AttentionGate(F_l=skip_ch, F_g=out_ch, F_int=max(out_ch // 2, 1))

        # after concat: out_ch + skip_ch
        self.res = ResidualBlock(out_ch + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)

        # if any off-by-1 due to odd sizes, center-crop skip to match x
        if skip.shape[-2:] != x.shape[-2:]:
            dh = skip.shape[-2] - x.shape[-2]
            dw = skip.shape[-1] - x.shape[-1]
            skip = skip[:, :, dh//2:skip.shape[-2]- (dh - dh//2),
                             dw//2:skip.shape[-1]- (dw - dw//2)]

        skip = self.att(skip, x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)


# -------------------------
# Attention ResUNet
# -------------------------
class AttentionResUNet(nn.Module):
    """
    Attention Res-U-Net matching your figure.

    Encoder: Residual blocks + maxpool
    Decoder: upconv + attention gate on skip + concat + residual block
    """
    def __init__(self, in_channels=1, n_classes=5, base_c=64):
        super().__init__()

        # Encoder
        self.enc1 = ResidualBlock(in_channels, base_c)          # 64
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ResidualBlock(base_c, base_c * 2)           # 128
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ResidualBlock(base_c * 2, base_c * 4)       # 256
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = ResidualBlock(base_c * 4, base_c * 8)       # 512
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ResidualBlock(base_c * 8, base_c * 16)  # 1024

        # Decoder (with attention on skips)
        self.up1 = UpBlockAttention(in_ch=base_c * 16, skip_ch=base_c * 8, out_ch=base_c * 8)
        self.up2 = UpBlockAttention(in_ch=base_c * 8,  skip_ch=base_c * 4, out_ch=base_c * 4)
        self.up3 = UpBlockAttention(in_ch=base_c * 4,  skip_ch=base_c * 2, out_ch=base_c * 2)
        self.up4 = UpBlockAttention(in_ch=base_c * 2,  skip_ch=base_c,     out_ch=base_c)

        # Output
        self.out_conv = nn.Conv2d(base_c, n_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        x1 = self.enc1(x)          # [B,64,H,W]
        p1 = self.pool1(x1)

        x2 = self.enc2(p1)         # [B,128,H/2,W/2]
        p2 = self.pool2(x2)

        x3 = self.enc3(p2)         # [B,256,H/4,W/4]
        p3 = self.pool3(x3)

        x4 = self.enc4(p3)         # [B,512,H/8,W/8]
        p4 = self.pool4(x4)

        # Bottleneck
        b = self.bottleneck(p4)    # [B,1024,H/16,W/16]

        # Decoder with attention-gated skips
        d1 = self.up1(b,  x4)      # [B,512,H/8,W/8]
        d2 = self.up2(d1, x3)      # [B,256,H/4,W/4]
        d3 = self.up3(d2, x2)      # [B,128,H/2,W/2]
        d4 = self.up4(d3, x1)      # [B,64,H,W]

        return self.out_conv(d4)   # [B,n_classes,H,W]
